## Level-wise (Depth-first) Growth

**How it works:**
- All nodes at depth *d* are split before any node at depth *d+1* is considered
- Creates a balanced tree structure
- Splits are evaluated globally across the entire level

**Pros:**
- **Regularization by design**: The balanced structure naturally limits tree complexity
- **Less prone to overfitting**: Harder to fit noise because you can't endlessly chase individual leaves
- **Faster training**: Fewer candidate splits to evaluate (you're not recursively diving into one branch)
- **More stable**: Splits are typically more "general" since they must compete with peers

**Cons:**
- May miss locally optimal splits deep in the tree
- Can underfit if the optimal structure is inherently imbalanced

---

## Leaf-wise (Best-first) Growth

**How it works:**
- At each iteration, you grow *the single leaf with the highest loss reduction* (best gain)
- Can create highly imbalanced, asymmetric trees
- Aggressively optimizes local regions

**Pros:**
- **Higher accuracy potential**: Directly minimizes training loss by always splitting the "worst" leaf
- **Efficient** for deep specialization: You can fit very specific patterns in the data
- Fewer total splits needed to achieve the same training loss

**Cons:**
- **Severe overfitting risk**: Unchecked, it can grow arbitrarily deep chasing noise
- **Requires careful regularization**: `max_depth`, `min_child_weight`, learning rate become critical
- **Slower per iteration**: More candidate splits to evaluate (you're exploring all leaves)

---

## Why Leaf-wise Overfits More Easily

1. **No global constraint**: Level-wise forces balance; leaf-wise doesn't
2. **Loss-chasing**: Optimizing for *any* leaf that reduces training loss means you'll eventually fit noise
3. **Sample size imbalance**: Deep leaf-wise trees can create leaves with very few samples—easy to overfit those regions
4. **Interaction effects**: Leaf-wise growth can capture high-order interactions that don't generalize

---

## Practical Example (SentinelPay context)

Imagine your fraud dataset has:
- A cluster of high-risk merchants (e.g., gift card resellers) where you can fit fraud patterns precisely
- Some legitimate merchants with rare edge cases that look like fraud

**Level-wise:**
- Grows a balanced tree; both clusters get equal "attention"
- May not capture the gift card fraud as sharply, but generalizes better

**Leaf-wise:**
- Aggressively splits the gift card leaf deeper and deeper
- Eventually fits noise within that cluster (specific IP ranges, transaction times that happen to correlate with fraud *in training* but not in production)
- Validation AUC might plateau or degrade

---

## XGBoost vs. LightGBM Strategy

| Aspect | XGBoost | LightGBM |
|--------|---------|----------|
| **Default Growth** | Level-wise | Leaf-wise |
| **Philosophy** | Safer by default; requires `max_depth` tuning | Aggressive by default; requires `num_leaves` / `max_depth` tuning |
| **Regularization Levers** | `max_depth`, `min_child_weight`, `subsample` | `num_leaves`, `min_data_in_leaf`, `lambda_l1/l2` |
| **Speed Trade-off** | Slower with deep trees | Faster, but easier to overfit |
